<a href="https://colab.research.google.com/github/SEC-API-io/sec-api-cookbook/blob/main/notebooks/xbrl-files/download-xbrl-files.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Download XBRL Data Files from SEC Filings

This example demonstrates how to find and download original XBRL data files attached to SEC filings, such as annual reports on Form 10-K.

> Note: XBRL data is also accessible in an aggregated and structured JSON format via the [XBRL-to-JSON API](https://sec-api.io/docs/xbrl-to-json-converter-api).

XBRL (eXtensible Business Reporting Language) files are XML-based documents that provide structured data for SEC EDGAR filings. They contain financial information like income statements, balance sheets, entity details (e.g., address, ticker symbol, auditor), and text blocks such as notes to financial statements. Many filings, such as annual and quarterly reports (Form 10-K, Form 10-Q), prospectuses (Form 424Bx) and registration statements (Form S-1, etc.), include an XBRL schema file that defines the structure of the filing data, along with other files that hold the actual structured data in XBRL format.

The metadata for these XBRL files, including their URLs and types, is found in the `dataFiles` array within the filing object returned by the Query API. Key fields such as `documentUrl`, `type`, and `description` provide details about each XBRL file, including its URL on EDGAR, the file type (e.g., `EX-101.INS`), and a description like `XBRL INSTANCE DOCUMENT`.

This example shows how to retrieve and download XBRL files attached to Form 10-Q filings from 2020 to 2023. The steps include:

1. Find and aggregate all URLs of the XBRL files attached to Form 10-Q filings.
2. Download all XBRL XML files retrieved in step 1 using the Filing Download API, and save them locally.

The example can be easily adapted to search for any filing form type, filer, or date range. For instance, it can be modified to locate XBRL files from 424B2 prospectuses.

In [ ]:
pip install sec-api

In [ ]:
SEC_API_KEY = "YOUR_API_KEY"

### Finding URLs of XBRL Files

To locate all EDGAR 10-Q filings that include XBRL data, a match-any search query such as `dataFiles:*` can be used. This query identifies any filing that contains a non-empty `dataFiles` array, as the array exclusively holds XBRL data. To further narrow down the search, a form type filter `formType:"10-Q" AND NOT formType:"10-Q/A"` is added, ensuring only 10-Q filings are retrieved, excluding amended versions. A date range filter is also applied to limit the search to a specific time frame, focusing on quarterly reports with XBRL data published within that period.

The final query looks like this:

```
dataFiles:* AND formType:"10-Q" AND NOT formType:"10-Q/A" AND filedAt:[2020-01-01 TO 2023-12-31]
```

Since the Query API can return a maximum of 10,000 filings per search query, and the number of 10-Q filings matching the above criteria exceeds 10,000, the query needs to be broken into smaller subsets. One approach is to construct search queries by month, iterating through the result pages for each month from 2020 to 2023. With a maximum of approximately 5,000 10-Q filings per month, the 10,000 result limit is never exceeded, allowing all filings to be fetched month by month, year over year.

In [ ]:
import pandas as pd
from sec_api import QueryApi

queryApi = QueryApi(SEC_API_KEY)

In [ ]:
filings = []

base_query = 'dataFiles:* AND formType:"10-Q" AND NOT formType:"10-Q/A"'

start_year = 2020
end_year = 2023

for year in range(start_year, end_year + 1):
    print(f"Starting to download metadata of filings from {year}")

    for month in range(1, 13):
        print(f"-- Starting month {month}")

        date_range_query = f"filedAt:[{year}-{month:02d}-01 TO {year}-{month:02d}-31]"
        query = f"{base_query} AND {date_range_query}"

        search_parameters = {
            "query": query,
            "from": 0,
            "size": 50,
            "sort": [{"filedAt": {"order": "desc"}}],
        }

        has_more_filings = True

        while has_more_filings:
            response = queryApi.get_filings(search_parameters)

            if len(response["filings"]) == 0:
                has_more_filings = False
                break

            filings.append(response["filings"])

            search_parameters["from"] += 50
            # uncomment the following line to fetch all filings
            break


filings = [item for sublist in filings for item in sublist]
filings = pd.DataFrame(filings)

Starting to download metadata of filings from 2022
-- Starting month 1
-- Starting month 2
-- Starting month 3
-- Starting month 4
-- Starting month 5
-- Starting month 6
-- Starting month 7
-- Starting month 8
-- Starting month 9
-- Starting month 10
-- Starting month 11
-- Starting month 12
Starting to download metadata of filings from 2023
-- Starting month 1
-- Starting month 2
-- Starting month 3
-- Starting month 4
-- Starting month 5
-- Starting month 6
-- Starting month 7
-- Starting month 8
-- Starting month 9
-- Starting month 10
-- Starting month 11
-- Starting month 12


In [ ]:
print(f"Total filings fetched: {len(filings)}")
print("10-Q filing metadata including XBRL data files:")
filings[["ticker", "cik", "formType", "accessionNo", "filedAt", "dataFiles"]].head(10)

Total filings fetched: 1200
10-Q filing metadata including XBRL data files:


,ticker,cik,formType,accessionNo,filedAt,dataFiles
0,EVOA,728447,10-Q,0000950170-22-000601,2022-01-31T19:24:32-05:00,"[{'sequence': '6', 'size': '125600', 'document..."
1,EVOA,728447,10-Q,0000950170-22-000600,2022-01-31T19:22:42-05:00,"[{'sequence': '6', 'size': '82410', 'documentU..."
2,EVOA,728447,10-Q,0000950170-22-000599,2022-01-31T19:20:34-05:00,"[{'sequence': '6', 'size': '862403', 'document..."
3,TVC,1376986,10-Q,0001376986-22-000005,2022-01-31T17:36:44-05:00,"[{'sequence': '6', 'size': '120761', 'document..."
4,HP,46765,10-Q,0000046765-22-000006,2022-01-31T17:23:47-05:00,"[{'sequence': '5', 'size': '59546', 'documentU..."
5,LUB,16099,10-Q,0000016099-22-000006,2022-01-31T16:52:26-05:00,"[{'sequence': '7', 'size': '50542', 'documentU..."
6,DLHC,785557,10-Q,0000785557-22-000003,2022-01-31T16:32:17-05:00,"[{'sequence': '5', 'size': '38022', 'documentU..."
7,CRUS,772406,10-Q,0000772406-22-000006,2022-01-31T16:01:19-05:00,"[{'sequence': '6', 'size': '36119', 'documentU..."
8,MNRO,876427,10-Q,0000876427-22-000003,2022-01-31T15:51:34-05:00,"[{'sequence': '6', 'size': '34194', 'documentU..."
9,ADP,8670,10-Q,0000008670-22-000014,2022-01-31T15:19:08-05:00,"[{'sequence': '8', 'size': '51863', 'documentU..."


In [ ]:
import json

print("Metadata of XBRL files from the first filing:")
print(json.dumps(response["filings"][0]["dataFiles"], indent=2))

Metadata of XBRL files from the first filing:
[
  {
    "sequence": "6",
    "size": "21999",
    "documentUrl": "https://www.sec.gov/Archives/edgar/data/1884072/000119983523000643/jewl-20230630.xsd",
    "description": "XBRL SCHEMA FILE",
    "type": "EX-101.SCH"
  },
  {
    "sequence": "7",
    "size": "36299",
    "documentUrl": "https://www.sec.gov/Archives/edgar/data/1884072/000119983523000643/jewl-20230630_cal.xml",
    "description": "XBRL CALCULATION FILE",
    "type": "EX-101.CAL"
  },
  {
    "sequence": "8",
    "size": "68475",
    "documentUrl": "https://www.sec.gov/Archives/edgar/data/1884072/000119983523000643/jewl-20230630_def.xml",
    "description": "XBRL DEFINITION FILE",
    "type": "EX-101.DEF"
  },
  {
    "sequence": "9",
    "size": "197267",
    "documentUrl": "https://www.sec.gov/Archives/edgar/data/1884072/000119983523000643/jewl-20230630_lab.xml",
    "description": "XBRL LABEL FILE",
    "type": "EX-101.LAB"
  },
  {
    "sequence": "10",
    "size": "1577

In [ ]:
xbrl_files = filings.explode("dataFiles")[
    ["ticker", "cik", "formType", "accessionNo", "filedAt", "dataFiles"]
]

columns_to_add = ["type", "description", "documentUrl"]

for col in columns_to_add:
    xbrl_files[col] = xbrl_files["dataFiles"].apply(
        lambda x: x[col] if col in x else None
    )

xbrl_files = xbrl_files.drop(columns=["dataFiles"])

# save to CSV file
xbrl_files.to_csv("xbrl_files.csv", index=False)

print("XBRL data files:")
xbrl_files

XBRL data files:


,ticker,cik,formType,accessionNo,filedAt,type,description,documentUrl
0,EVOA,728447,10-Q,0000950170-22-000601,2022-01-31T19:24:32-05:00,EX-101.SCH,XBRL TAXONOMY EXTENSION SCHEMA DOCUMENT,https://www.sec.gov/Archives/edgar/data/728447...
0,EVOA,728447,10-Q,0000950170-22-000601,2022-01-31T19:24:32-05:00,EX-101.PRE,XBRL TAXONOMY EXTENSION PRESENTATION LINKBASE ...,https://www.sec.gov/Archives/edgar/data/728447...
0,EVOA,728447,10-Q,0000950170-22-000601,2022-01-31T19:24:32-05:00,EX-101.LAB,XBRL TAXONOMY EXTENSION LABEL LINKBASE DOCUMENT,https://www.sec.gov/Archives/edgar/data/728447...
0,EVOA,728447,10-Q,0000950170-22-000601,2022-01-31T19:24:32-05:00,EX-101.CAL,XBRL TAXONOMY EXTENSION CALCULATION LINKBASE D...,https://www.sec.gov/Archives/edgar/data/728447...
0,EVOA,728447,10-Q,0000950170-22-000601,2022-01-31T19:24:32-05:00,EX-101.DEF,XBRL TAXONOMY EXTENSION DEFINITION LINKBASE DO...,https://www.sec.gov/Archives/edgar/data/728447...
...,...,...,...,...,...,...,...,...
1199,AITR,1966734,10-Q,0001493152-23-045528,2023-12-20T11:13:54-05:00,EX-101.CAL,XBRL CALCULATION FILE,https://www.sec.gov/Archives/edgar/data/196673...
1199,AITR,1966734,10-Q,0001493152-23-045528,2023-12-20T11:13:54-05:00,EX-101.DEF,XBRL DEFINITION FILE,https://www.sec.gov/Archives/edgar/data/196673...
1199,AITR,1966734,10-Q,0001493152-23-045528,2023-12-20T11:13:54-05:00,EX-101.LAB,XBRL LABEL FILE,https://www.sec.gov/Archives/edgar/data/196673...
1199,AITR,1966734,10-Q,0001493152-23-045528,2023-12-20T11:13:54-05:00,EX-101.PRE,XBRL PRESENTATION FILE,https://www.sec.gov/Archives/edgar/data/196673...


### Download XBRL Files from SEC Filings

In the final step, the XBRL files from SEC filings are downloaded and organized into the following folder structure: `xbrl-files/<cik>/<accessionNo>/<edgar_file_type>.<file_extension>`, where `<file_extension>` refers to the XBRL file type (e.g., `xml` or `xsd`). An example folder structure is shown below:

```
xbrl-files/
    320193/
        0000320193-21-000139/
            EX-101.SCH.xsd
            EX-101.CAL.xml
            EX-101.DEF.xml
            EX-101.LAB.xml
            EX-101.PRE.xml
            XML.xml
    .../
```

The following table provides an overview of the XBRL file types and their descriptions:

| EDGAR File Type | File Extension | Description |
| --- | --- | --- |
| EX-101.SCH | *.xsd | XBRL Taxonomy Schema |
| EX-101.CAL | *.xml | XBRL Calculation Linkbase |
| EX-101.DEF | *.xml | XBRL Definition Linkbase |
| EX-101.LAB | *.xml | XBRL Label Linkbase |
| EX-101.PRE | *.xml | XBRL Presentation Linkbase |
| XML | *.xml | XBRL Instance Document |

To efficiently download multiple XBRL files at once, the `pandarallel` package is used to parallelize the download process across multiple threads, significantly speeding up the retrieval process.

In [ ]:
pip install pandarallel ipywidgets

In [ ]:
import os
from pandarallel import pandarallel
from sec_api import RenderApi

pandarallel.initialize(nb_workers=4, progress_bar=True)

renderApi = RenderApi(SEC_API_KEY)

INFO: Pandarallel will run on 4 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.


In [ ]:
def download_and_save_xbrl_file(row):
    cik = row["cik"]
    accessionNo = row["accessionNo"]
    file_url = row["documentUrl"]
    file_type = row["type"]
    filename_type = file_url.split(".")[-1]

    try:
        xbrl_data = renderApi.get_file(file_url)

        xbrl_file_name = f"{file_type}.{filename_type}"
        folder_path = f"xbrl-files/{cik}/{accessionNo}"
        file_path = f"{folder_path}/{xbrl_file_name}"

        if not os.path.exists(folder_path):
            os.makedirs(folder_path)

        with open(file_path, "w") as f:
            f.write(xbrl_data)

    except Exception as e:
        print(f"Failed to download {file_url} for {cik} - {accessionNo}\n")
        return None


# download and save the first 50 XBRL files
results = xbrl_files[:50].parallel_apply(download_and_save_xbrl_file, axis=1)
# uncomment the line below to download all XBRL files
# results = xbrl_files.parallel_apply(download_and_save_xbrl_file, axis=1)

print(f"Downloaded {len(results)} XBRL files")

Downloaded 50 XBRL files
